In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import xgboost as xgb

# pip install torch==2.* scikit-learn
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression



In [2]:
X_train = pd.read_csv('data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('data/sample_submission.csv',index_col='ROW_ID')

In [3]:
RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']

In [4]:
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i+1]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[ f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i+1]].mean(1)
    X_test[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

In [5]:
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]

In [6]:
ret_cols = [f"RET_{i}" for i in range(1, 21)]
r = X_train[ret_cols].to_numpy()

rtest = X_test[ret_cols].to_numpy()

def ema(arr, L):
    alpha = 2/(L+1)
    w = (1-alpha) ** np.arange(L)  # 0..L-1
    w = w / w.sum()
    return (arr[:, :L] * w).sum(axis=1)

X_train["ema3"]  = ema(r, 3)
X_train["ema5"]  = ema(r, 5)
X_train["ema10"] = ema(r,10)


X_test["ema3"]  = ema(rtest, 3)
X_test["ema5"]  = ema(rtest, 5)
X_test["ema10"] = ema(rtest,10)

m20 = r.mean(axis=1)
s20 = r.std(axis=1, ddof=0)

m20test = rtest.mean(axis=1)
s20test = rtest.std(axis=1, ddof=0)

X_train["z20"] = m20 / (s20 + 1e-12)
X_test["z20"] = m20test / (s20test + 1e-12)

In [7]:
sign = np.sign(r)  
signtest = np.sign(rtest)

def last_streak_len(sig_row, positive=True):
    # part de RET_1 vers RET_20
    target = 1 if positive else -1
    cnt = 0
    for v in sig_row[:20]:  # [:20] explicite
        if v == target:
            cnt += 1
        else:
            break
    return cnt

X_train["streak_pos"] = [last_streak_len(s, True) for s in sign]
X_train["streak_neg"] = [last_streak_len(s, False) for s in sign]

X_test["streak_pos"] = [last_streak_len(s, True) for s in signtest]
X_test["streak_neg"] = [last_streak_len(s, False) for s in signtest]

p = (r > 0).mean(axis=1)
p = np.clip(p, 1e-9, 1 - 1e-9)
X_train["sign_entropy20"] = -(p*np.log(p) + (1-p)*np.log(1-p))

ptest = (rtest > 0).mean(axis=1)    
ptest = np.clip(ptest, 1e-9, 1 - 1e-9)
X_test["sign_entropy20"] = -(ptest*np.log(ptest) + (1-ptest)*np.log(1-ptest))


In [8]:
features = [col for col in X_train.columns if col not in ['TS', 'ALLOCATION']]

# Autoencodeur

In [9]:

# ==== Prépa data ====
features = [c for c in X_train.columns if c not in ["TS","ALLOCATION"]]  # adapte si besoin
y_cont = y_train["target"].astype(float).values
y_sign = (y_cont > 0).astype(int)

# IDs d'allocations (fixes sur 65)
alloc2id = X_train["ALLOCATION"].astype("category").cat.codes.values  # 0..64
ts_all = X_train["TS"].values

device = torch.device("cpu")

# ==== Modules ====
class Encoder(nn.Module):
    def __init__(self, d_in, d_lat=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, d_lat)
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self, d_lat, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_lat, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, d_in)
        )
    def forward(self, z): return self.net(z)

class DAE(nn.Module):
    def __init__(self, d_in, d_lat=16, sigma=0.1):
        super().__init__()
        self.sigma = sigma
        self.enc = Encoder(d_in, d_lat)
        self.dec = Decoder(d_lat, d_in)
        self.cls = nn.Linear(d_lat, 1)     # BCE (option)
        self.reg = nn.Linear(d_lat, 1)     # MSE
    def forward(self, x):
        x_tilde = x + self.sigma*torch.randn_like(x)
        z = self.enc(x_tilde)
        x_hat = self.dec(z)
        p = torch.sigmoid(self.cls(z))
        y_reg = self.reg(z)
        return z, x_hat, p, y_reg
    @torch.no_grad()
    def encode_clean(self, x):  # sans bruit, pour extraire z
        return self.enc(x)

class MLPFinal(nn.Module):
    def __init__(self, d_in, d_lat, n_alloc=65, d_alloc=6):
        super().__init__()
        self.emb = nn.Embedding(n_alloc, d_alloc)
        self.net = nn.Sequential(
            nn.Linear(d_in + d_lat + d_alloc, 128), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)  # régression; signe via >0
        )
    def forward(self, x, z, alloc_id):
        e = self.emb(alloc_id)
        h = torch.cat([x, z, e], dim=1)
        return self.net(h)

# ==== Dataset torch (par ligne) ====
class TabDS(Dataset):
    def __init__(self, X, y, alloc_id, w=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1,1)
        self.a = torch.tensor(alloc_id, dtype=torch.long)
        self.w = None if w is None else torch.tensor(w, dtype=torch.float32).view(-1,1)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): 
        if self.w is None: return self.X[i], self.y[i], self.a[i]
        return self.X[i], self.y[i], self.a[i], self.w[i]



In [ ]:

def train_dae(dae, dl, epochs=10, lam_rec=1.0, lam_reg=1.0, lam_cls=1.0, lr=1e-3):
    dae.train()
    opt = torch.optim.AdamW(dae.parameters(), lr=lr, weight_decay=1e-4)
    mse = nn.MSELoss()
    bce = nn.BCELoss()
    for _ in range(epochs):
        for batch in dl:
            if len(batch)==3:
                xb, yb, _ab = batch; wb=None
            else:
                xb, yb, _ab, wb = batch
            xb, yb = xb.to(device), yb.to(device)
            z, x_hat, p, y_reg = dae(xb)
            loss = lam_rec*mse(x_hat, xb) + lam_reg*mse(y_reg, yb) + lam_cls*bce(p, (yb>0).float())
            opt.zero_grad(); loss.backward(); opt.step()

def train_mlp(mlp, dl, dae, epochs=10, lr=1e-3):
    mlp.train(); dae.eval()
    opt = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=1e-4)
    mse = nn.MSELoss(reduction="none")
    for _ in range(epochs):
        for batch in dl:
            if len(batch)==3:
                xb, yb, ab = batch; wb=None
            else:
                xb, yb, ab, wb = batch
            xb, yb, ab = xb.to(device), yb.to(device), ab.to(device)
            with torch.no_grad():
                z = dae.encode_clean(xb)
            y_hat = mlp(xb, z, ab)
            loss_el = mse(y_hat, yb)
            if wb is not None:
                wb = wb.to(device)
                loss = (loss_el * wb).mean()
            else:
                loss = loss_el.mean()
            opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def predict_scores(mlp, dae, X, alloc_id, bs=4096):
    mlp.eval(); dae.eval()
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    A_t = torch.tensor(alloc_id, dtype=torch.long, device=device)
    out = []
    for i in range(0, len(X), bs):
        xb = X_t[i:i+bs]
        ab = A_t[i:i+bs]
        z = dae.encode_clean(xb)
        y = mlp(xb, z, ab)
        out.append(y.squeeze(1).cpu().numpy())
    return np.concatenate(out)


In [11]:
X_train

,TS,ALLOCATION,RET_20,RET_19,RET_18,RET_17,RET_16,RET_15,RET_14,RET_13,...,ALLOCATIONS_AVERAGE_PERF_15,AVERAGE_PERF_20,ALLOCATIONS_AVERAGE_PERF_20,ema3,ema5,ema10,z20,streak_pos,streak_neg,sign_entropy20
ROW_ID,,,,,,,,,,,,,,,,,,,,,
0,DATE_0001,ALLOCATION_01,-0.002477,0.004826,0.005374,-0.001688,-0.000152,-0.000685,-0.002217,0.001911,...,0.000230,0.000507,0.000087,0.001056,0.000689,0.000353,0.103721,2,0,0.673012
1,DATE_0001,ALLOCATION_02,0.006863,-0.005265,-0.004249,0.002686,-0.002638,0.003056,0.002712,-0.005269,...,0.000230,-0.000701,0.000087,-0.002584,-0.001911,-0.000899,-0.081951,0,2,0.693147
2,DATE_0001,ALLOCATION_03,-0.005535,0.008541,0.005360,-0.002491,0.004679,-0.000848,-0.007197,0.006792,...,0.000230,0.001370,0.000087,0.003220,0.002089,0.001501,0.187398,2,0,0.688139
3,DATE_0001,ALLOCATION_04,0.003178,-0.001352,-0.004051,-0.001841,-0.005659,0.000627,0.006686,0.001804,...,0.000230,-0.001050,0.000087,-0.003191,-0.003086,-0.002344,-0.257525,0,2,0.688139
4,DATE_0001,ALLOCATION_05,0.003359,-0.003349,-0.005460,0.000416,-0.003533,0.000913,0.005088,-0.003043,...,0.000230,-0.000607,0.000087,-0.000913,-0.000661,-0.000467,-0.149394,0,2,0.688139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180240,DATE_2773,ALLOCATION_61,0.002236,-0.000891,0.001505,-0.001188,0.001114,-0.000919,-0.002004,0.001065,...,-0.000104,0.000201,-0.000074,0.000545,0.000471,0.000381,0.289738,2,0,0.647447
180241,DATE_2773,ALLOCATION_62,0.001811,0.003863,-0.000609,0.000014,-0.000569,0.002425,-0.004095,-0.005187,...,-0.000104,-0.000370,-0.000074,-0.001870,-0.001475,-0.000830,-0.096274,0,1,0.688139
180242,DATE_2773,ALLOCATION_63,0.001063,-0.000599,0.001783,-0.001443,0.001426,-0.001503,0.000086,0.002414,...,-0.000104,-0.000190,-0.000074,-0.000365,-0.000198,-0.000393,-0.096068,0,1,0.673012


In [10]:

# ==== CV par TS ====
d_in = len(features)
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
uniq_ts = X_train["TS"].unique()

oof_scores = np.full(len(X_train), np.nan, dtype=float)


In [11]:

for fold, (itr, iva) in enumerate(kf.split(uniq_ts), 1):
    tr_ts, va_ts = uniq_ts[itr], uniq_ts[iva]
    m_tr = np.isin(ts_all, tr_ts)
    m_va = np.isin(ts_all, va_ts)

    X_tr_raw = X_train.loc[m_tr, features].to_numpy(np.float32)
    X_va_raw = X_train.loc[m_va, features].to_numpy(np.float32)
    y_tr, y_va = y_cont[m_tr], y_cont[m_va]
    a_tr, a_va = alloc2id[m_tr], alloc2id[m_va]

    # standardisation par fold
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr_raw)
    X_va = sc.transform(X_va_raw)

    # poids d'exemple ∝ |y|
    w_tr = np.abs(y_tr)
    w_tr = w_tr / (w_tr.mean() + 1e-12)

    print("Made it to data loader")

    # Dataloaders
    ds_dae = TabDS(X_tr, y_tr, a_tr)                 # DAE : pas besoin des poids ici
    dl_dae = DataLoader(ds_dae, batch_size=256, shuffle=True)

    ds_mlp = TabDS(X_tr, y_tr, a_tr, w_tr)           # MLP : on met les poids
    dl_mlp = DataLoader(ds_mlp, batch_size=256, shuffle=True)

    print("Made it to models")
    # init modèles
    dae = DAE(d_in=d_in, d_lat=16, sigma=0.1).to(device)

    print("DAE worked")
    mlp = MLPFinal(d_in=d_in, d_lat=16, n_alloc=65, d_alloc=6).to(device)

    # train DAE (multi-tâches) + MLP final
    train_dae(dae, dl_dae, epochs=12, lam_rec=1.0, lam_reg=1.0, lam_cls=1.0, lr=1e-3)
    train_mlp(mlp, dl_mlp, dae, epochs=10, lr=1e-3)

    # OOF scores (continus) sur val
    oof_scores[m_va] = predict_scores(mlp, dae, X_va, a_va)

    acc = accuracy_score((y_va>0), (oof_scores[m_va]>0))
    print(f"Fold {fold:02d} acc={acc*100:.2f}%  (n_val={m_va.sum()})")



: 

In [ ]:
# ==== Calibration isotone + seuil optimal (sur OOF) ====
ir = IsotonicRegression(out_of_bounds="clip")
ir.fit(oof_scores, y_sign)
p_oof = ir.transform(oof_scores)

thr_grid = np.linspace(0.3, 0.7, 401)
best_thr, best_acc = max(
    ((t, accuracy_score(y_sign, p_oof >= t)) for t in thr_grid),
    key=lambda x: x[1]
)
print(f"[OOF] isotonic acc={best_acc*100:.2f}% @thr={best_thr:.3f}")